## Step 1. Imports

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import StructType

from notebooks._shared.configuration import AppConfig
from notebooks._shared.contracts import BRONZE_MOVIES, BRONZE_CREDITS

## Step 2. Functions

In [0]:
def bronze_schema_without_metadata(contract):
    metadata_columns = {
        "_source_file",
        "_ingested_at",
        "_ingestion_id",
    }

    return StructType(
        [
            field
            for field in contract.schema.fields
            if field.name not in metadata_columns
        ]
    )


def validate_header(actual_header, expected_schema, source_name):
    expected_header = [field.name for field in expected_schema.fields]

    if actual_header != expected_header:
        raise ValueError(
            f"Header divergente para {source_name}.\n"
            f"Actual:   {actual_header}\n"
            f"Expected: {expected_header}"
        )


def validate_schema_against_contract(df, contract):
    actual_fields = df.schema.fields
    expected_fields = contract.schema.fields

    if len(actual_fields) != len(expected_fields):
        raise ValueError(
            f"Qtd de campos divergentes para {contract.name}: "
            f"actual={len(actual_fields)}, expected={len(expected_fields)}"
        )

    differences = []

    for position, (actual, expected) in enumerate(
        zip(actual_fields, expected_fields),
        start=1,
    ):
        if actual.name != expected.name:
            differences.append(
                f"posição {position}: "
                f"name actual={actual.name}, expected={expected.name}"
            )

        if actual.dataType != expected.dataType:
            differences.append(
                f"posição {position} ({expected.name}): "
                f"type actual={actual.dataType.simpleString()}, "
                f"expected={expected.dataType.simpleString()}"
            )

    if differences:
        raise ValueError(
            f"Schema divergente para {contract.name}:\n" + "\n".join(differences)
        )

## Step 3. Parâmetros de execução

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("raw_volume", "")

## Step 4. Configuração

In [0]:
config = AppConfig(
    catalog=dbutils.widgets.get("catalog"),
    bronze_schema=dbutils.widgets.get("bronze_schema"),
    raw_volume=dbutils.widgets.get("raw_volume"),
)

movies_source_path = f"{config.raw_volume_path}/tmdb_5000_movies.csv"
credits_source_path = f"{config.raw_volume_path}/tmdb_5000_credits.csv"

movies_source_schema = bronze_schema_without_metadata(BRONZE_MOVIES)
credits_source_schema = bronze_schema_without_metadata(BRONZE_CREDITS)

movies_table = f"{config.bronze_namespace}.{BRONZE_MOVIES.name}"
credits_table = f"{config.bronze_namespace}.{BRONZE_CREDITS.name}"

## Step 5. Validação dos headers

In [0]:
movies_header = (
    spark.read.option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(movies_source_path)
    .columns
)

credits_header = (
    spark.read.option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(credits_source_path)
    .columns
)

validate_header(
    movies_header,
    movies_source_schema,
    BRONZE_MOVIES.name,
)

validate_header(
    credits_header,
    credits_source_schema,
    BRONZE_CREDITS.name,
)

## Step 6. Leitura dos CSVs

In [0]:
movies_df = (
    spark.read.option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(movies_source_schema)
    .csv(movies_source_path)
)

credits_df = (
    spark.read.option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(credits_source_schema)
    .csv(credits_source_path)
)

## Step 7. Metadados

In [0]:
ingestion_id = str(uuid4())
ingested_at = datetime.now(timezone.utc)

movies_bronze_df = (
    movies_df
    .withColumn("_source_file", F.lit(movies_source_path))
    .withColumn("_ingested_at", F.lit(ingested_at))
    .withColumn("_ingestion_id", F.lit(ingestion_id))
)

credits_bronze_df = (
    credits_df
    .withColumn("_source_file", F.lit(credits_source_path))
    .withColumn("_ingested_at", F.lit(ingested_at))
    .withColumn("_ingestion_id", F.lit(ingestion_id))
)

## Step 8. Validação estrutural

In [0]:
validate_schema_against_contract(
    movies_bronze_df,
    BRONZE_MOVIES,
)

validate_schema_against_contract(
    credits_bronze_df,
    BRONZE_CREDITS,
)

## Step 9. Escrita Bronze

In [0]:
(
    movies_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(movies_table)
)

(
    credits_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(credits_table)
)

## Step 10. Finalização

In [0]:
print("Bronze ingestion completed sucessfully")
print(f"Ingestion ID: {ingestion_id}")
print(f"Ingested at: {ingested_at}")
print(f"Movies Table: {movies_table}")
print(f"Credits Table: {credits_table}")